In [2]:
import pandas as pd
import numpy as np

import shap

import matplotlib
matplotlib.use('Agg')

import matplotlib.pyplot as plt

import joblib
import json
import warnings
import os

warnings.filterwarnings('ignore')

os.makedirs('../plots', exist_ok=True)

# ---------------------------------------------------
# Load Dataset
# ---------------------------------------------------

df = pd.read_csv(
    '../data/student_loans_engineered.csv'
)

# ---------------------------------------------------
# Load Metrics + Features
# ---------------------------------------------------

metrics = json.load(
    open('../models/model_metrics.json')
)

FEATURES = metrics['features']

# ---------------------------------------------------
# Load 90-Day Model
# ---------------------------------------------------

model_90d = joblib.load(
    '../models/model_default_90d.pkl'
)

# ---------------------------------------------------
# Train/Test Split
# ---------------------------------------------------

from sklearn.model_selection import train_test_split

X = df[FEATURES]
y = df['default_90d']

_, X_test, _, _ = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Computing SHAP for 90-day model...")

# ---------------------------------------------------
# SHAP Explainer
# ---------------------------------------------------

explainer = shap.TreeExplainer(model_90d)

X_sample = X_test.iloc[:500]

shap_values = explainer.shap_values(X_sample)

sv = shap_values[1] if isinstance(
    shap_values,
    list
) else shap_values

# ---------------------------------------------------
# SHAP BAR PLOT
# ---------------------------------------------------

plt.figure(figsize=(10, 7))

shap.summary_plot(
    sv,
    X_sample,
    feature_names=FEATURES,
    plot_type='bar',
    show=False
)

plt.title(
    'Feature Importance - 90-Day Default (SHAP)',
    fontweight='bold'
)

plt.tight_layout()

plt.savefig(
    '../plots/shap_90d_bar.png',
    dpi=150,
    bbox_inches='tight'
)

plt.close()

print("Saved: plots/shap_90d_bar.png")

# ---------------------------------------------------
# SHAP BEESWARM PLOT
# ---------------------------------------------------

plt.figure(figsize=(10, 8))

shap.summary_plot(
    sv,
    X_sample,
    feature_names=FEATURES,
    plot_type='dot',
    max_display=15,
    show=False
)

plt.title(
    'SHAP Beeswarm - 90-Day Model',
    fontweight='bold'
)

plt.tight_layout()

plt.savefig(
    '../plots/shap_90d_beeswarm.png',
    dpi=150,
    bbox_inches='tight'
)

plt.close()

print("Saved: plots/shap_90d_beeswarm.png")

# ---------------------------------------------------
# Compare SHAP Across All Horizons
# ---------------------------------------------------

mean_abs = {}

for horizon in [
    'default_30d',
    'default_60d',
    'default_90d'
]:

    m = joblib.load(
        f'../models/model_{horizon}.pkl'
    )

    e = shap.TreeExplainer(m)

    sv_h = e.shap_values(X_sample)

    sv_h = sv_h[1] if isinstance(
        sv_h,
        list
    ) else sv_h

    mean_abs[horizon] = np.abs(
        sv_h
    ).mean(axis=0)

# ---------------------------------------------------
# Importance DataFrame
# ---------------------------------------------------

importance_df = pd.DataFrame(
    mean_abs,
    index=FEATURES
).sort_values(
    'default_90d',
    ascending=False
)

top10 = importance_df.head(10)

# ---------------------------------------------------
# Comparison Plot
# ---------------------------------------------------

fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(top10))
w = 0.25

ax.bar(
    x - w,
    top10['default_30d'],
    width=w,
    label='30-Day',
    color='#16A34A',
    alpha=0.85
)

ax.bar(
    x,
    top10['default_60d'],
    width=w,
    label='60-Day',
    color='#EAB308',
    alpha=0.85
)

ax.bar(
    x + w,
    top10['default_90d'],
    width=w,
    label='90-Day',
    color='#EF4444',
    alpha=0.85
)

ax.set_xticks(x)

ax.set_xticklabels(
    top10.index,
    rotation=45,
    ha='right',
    fontsize=9
)

ax.set_title(
    'Top 10 Feature Importance - Across All 3 Horizons',
    fontweight='bold'
)

ax.legend()

plt.tight_layout()

plt.savefig(
    '../plots/shap_all_horizons_comparison.png',
    dpi=150,
    bbox_inches='tight'
)

plt.close()

print("Saved: plots/shap_all_horizons_comparison.png")

# ---------------------------------------------------
# Top Features
# ---------------------------------------------------

print()
print("TOP 5 FEATURES FOR 90-DAY MODEL:")

for feat, val in importance_df[
    'default_90d'
].head(5).items():

    print(f"{feat:25}: {val:.4f}")

D:\Student loan warning\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Computing SHAP for 90-day model...
Saved: plots/shap_90d_bar.png
Saved: plots/shap_90d_beeswarm.png
Saved: plots/shap_all_horizons_comparison.png

TOP 5 FEATURES FOR 90-DAY MODEL:
missed_pmts_past         : 0.5179
employed                 : 0.4848
academic_risk            : 0.4128
tier_enc                 : 0.2421
support_score            : 0.2014
